In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('cleaned_data.csv')
df.sample()

,age,RevolvingUtilizationOfUnsecuredLines,NumberOfTime30-59DaysPastDueNotWorse,NumberOfTime60-89DaysPastDueNotWorse,NumberOfTimes90DaysLate,DebtRatio,DebtRatioMissing,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberRealEstateLoansOrLines,NumberOfDependents,SeriousDlqin2yrs
39946,49,0.58611,0,0,0,0.111233,0,6400.0,4,0,0.0,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134206 entries, 0 to 134205
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   age                                   134206 non-null  int64  
 1   RevolvingUtilizationOfUnsecuredLines  134206 non-null  float64
 2   NumberOfTime30-59DaysPastDueNotWorse  134206 non-null  int64  
 3   NumberOfTime60-89DaysPastDueNotWorse  134206 non-null  int64  
 4   NumberOfTimes90DaysLate               134206 non-null  int64  
 5   DebtRatio                             134206 non-null  float64
 6   DebtRatioMissing                      134206 non-null  int64  
 7   MonthlyIncome                         134206 non-null  float64
 8   NumberOfOpenCreditLinesAndLoans       134206 non-null  int64  
 9   NumberRealEstateLoansOrLines          134206 non-null  int64  
 10  NumberOfDependents                    134206 non-null  float64
 11  

In [4]:
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [5]:
from sklearn.model_selection import train_test_split

In [6]:
X_train, X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42)

In [7]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

In [8]:
from sklearn.model_selection import cross_val_score

In [9]:
import optuna

In [10]:
from imblearn.over_sampling import SMOTENC

In [11]:
smotenc = SMOTENC(categorical_features=[6],random_state=42)

In [12]:
X_train_resampled, y_train_resampled = smotenc.fit_resample(X_train, y_train)

In [13]:
y_train_resampled.value_counts()

SeriousDlqin2yrs
0    94585
1    94585
Name: count, dtype: int64

In [14]:
def objective(trial):

    classifier_model = trial.suggest_categorical('classifier',['RFC','GB','Logistic'])

    if classifier_model=='Logistic':
        C = trial.suggest_float('logistic_C',0.1,100,log=True)
        class_weight = trial.suggest_categorical('class_weight', [None, 'balanced'])
        solver = trial.suggest_categorical('solver',['lbfgs','liblinear'])
        model = LogisticRegression(C=C,class_weight=class_weight,solver=solver,max_iter=1000)

    elif classifier_model=='RFC':
        n_estimators = trial.suggest_int('estimators',20,150,step=2)
        criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
        max_depth = trial.suggest_int('max_depth',3,7)
        min_samples_split = trial.suggest_int('min_samples_split',2,10,step=2)
        min_samples_leaf = trial.suggest_int('min_samples_leaf',2,16,step=2)
        max_features = trial.suggest_categorical('max_features',['sqrt','log2'])
        bootstrap = trial.suggest_categorical('bootstrap',[True,False])

        model = RandomForestClassifier(n_estimators=n_estimators,criterion=criterion,max_depth=max_depth,min_samples_split=min_samples_split,
                                      min_samples_leaf=min_samples_leaf,max_features=max_features,bootstrap=bootstrap,random_state=42)


    elif classifier_model == 'GB':

        n_estimators=trial.suggest_int('n_estimators',50,150,step=2)
        learning_rate=trial.suggest_float('learning_rate',0.01,0.3,log=True)
        min_samples_split = trial.suggest_int('min_samples_split',2,10,step=2)
        min_samples_leaf = trial.suggest_int('min_samples_leaf',2,16,step=2)

        model = GradientBoostingClassifier(n_estimators=n_estimators,learning_rate=learning_rate,
                                           min_samples_split=min_samples_split,min_samples_leaf=min_samples_leaf,random_state=42)

    
    score = cross_val_score(model, X_train_resampled, y_train_resampled, cv=3, scoring='recall').mean()    
    return score  # Return the recall score for Optuna to maximize

In [15]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize recall
study.optimize(objective, n_trials=125)  # Run 125 trials to find the best hyperparameters

[I 2026-08-18 07:56:45,741] A new study created in memory with name: no-name-daa16564-b59e-421c-bbc8-81b509eec25e
[I 2026-08-18 07:57:29,494] Trial 0 finished with value: 0.8506632861320181 and parameters: {'classifier': 'RFC', 'estimators': 110, 'criterion': 'entropy', 'max_depth': 5, 'min_samples_split': 10, 'min_samples_leaf': 16, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 0 with value: 0.8506632861320181.
[I 2026-08-18 07:57:31,937] Trial 1 finished with value: 0.7465983043655213 and parameters: {'classifier': 'Logistic', 'logistic_C': 1.9383703036398403, 'class_weight': None, 'solver': 'liblinear'}. Best is trial 0 with value: 0.8506632861320181.
[I 2026-08-18 07:58:14,103] Trial 2 finished with value: 0.8510121984470814 and parameters: {'classifier': 'RFC', 'estimators': 64, 'criterion': 'entropy', 'max_depth': 5, 'min_samples_split': 6, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 2 with value: 0.8510121984470814.
[I 2026-08-18 

KeyboardInterrupt: 

In [16]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.8510544825266319
Best hyperparameters: {'classifier': 'RFC', 'estimators': 96, 'criterion': 'gini', 'max_depth': 5, 'min_samples_split': 4, 'min_samples_leaf': 12, 'max_features': 'sqrt', 'bootstrap': True}


In [17]:
Best_model = RandomForestClassifier(n_estimators=96,criterion='gini',max_depth=5,min_samples_split=4,min_samples_leaf=12,
                                    max_features='sqrt',bootstrap=True)

In [1]:
from imblearn.pipeline import Pipeline    # Not using sklearn's pipeline, because it does not work with SMOTENC

In [19]:
final_pipe = Pipeline(
    [
     ('SMOTENC',smotenc),
     ('Best Model',Best_model)
    ]
)

In [21]:
final_pipe.fit(X_train, y_train)

Pipeline(steps=[('Imbalance Handling',
                 SMOTENC(categorical_features=[6], random_state=42)),
                ('Best Model',
                 RandomForestClassifier(max_depth=5, min_samples_leaf=12,
                                        min_samples_split=4,
                                        n_estimators=96))])

In [22]:
y_preds = final_pipe.predict(X_test)

In [24]:
y_preds.shape

(33552,)

In [25]:
X_test.shape

(33552, 11)

In [26]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, recall_score

In [28]:
accuracy_score(y_test,y_preds)

0.7344122556032427

In [29]:
recall_score(y_test,y_preds)

0.7227926078028748

In [30]:
print('Classification report for Logistic with SMOTENC')
print(classification_report(y_test,y_preds))

Classification report for Logistic with SMOTENC
              precision    recall  f1-score   support

           0       0.98      0.74      0.84     31604
           1       0.14      0.72      0.24      1948

    accuracy                           0.73     33552
   macro avg       0.56      0.73      0.54     33552
weighted avg       0.93      0.73      0.80     33552

